# imports

In [9]:
from collections import defaultdict
import os
import glob
import yaml
import os
import glob
import shutil


# Análise dos datasets separados

In [ ]:
# ==========================================
# CONFIGURAÇÕES
# ==========================================
PASTAS_ALVO = [
    'Recyclable Material Sorting ver2.v1i.yolov8',  # Dataset Base com +12k imagens
    'waste detection.v2i.yolov8',                   # Dataset com mais imagens para complemento para todas as classes
    'waste detection.v8i.yolov8'                    # Dataset mais diversificado sem a classe de papelão
]

SPLITS = ['train', 'valid', 'test']

def contar_objetos_com_nomes_do_yaml(pastas):
    contagem_total_geral = defaultdict(int)
    nomes_gerais_referencia = {} # Guarda os nomes da primeira pasta válida para o resumo final

    print("="*60)
    print(" CONTAGEM DE OBJETOS MAPEANDO COM DATA.YAML")
    print("="*60)

    for pasta in pastas:
        contagem_pasta = defaultdict(int)
        arquivos_analisados = 0
        nomes_classes = {}
        
        if not os.path.exists(pasta):
            print(f"\n[!] Aviso: A pasta '{pasta}' não foi encontrada.")
            continue

        #  Busca e lê o arquivo data.yaml na raiz da pasta
        caminho_yaml = os.path.join(pasta, 'data.yaml')
        if os.path.exists(caminho_yaml):
            with open(caminho_yaml, 'r', encoding='utf-8') as f:
                dados_yaml = yaml.safe_load(f)
                # O Roboflow salva os nomes na chave 'names'
                names = dados_yaml.get('names', [])
                
                # Trata se for uma lista ou dicionário
                if isinstance(names, dict):
                    nomes_classes = names
                elif isinstance(names, list):
                    nomes_classes = {i: nome for i, nome in enumerate(names)}
                
                # Salva os nomes da primeira pasta para usar no resumo final
                if not nomes_gerais_referencia and nomes_classes:
                    nomes_gerais_referencia = nomes_classes
        else:
            print(f"\n[!] Aviso: data.yaml não encontrado em '{pasta}'. As classes ficarão sem nome.")

        # Conta os objetos nos arquivos .txt
        for split in SPLITS:
            caminho_labels = os.path.join(pasta, split, 'labels')
            if not os.path.exists(caminho_labels):
                continue
                
            arquivos_txt = glob.glob(os.path.join(caminho_labels, '*.txt'))
            arquivos_analisados += len(arquivos_txt)
            
            for txt in arquivos_txt:
                with open(txt, 'r') as f:
                    linhas = f.readlines()
                    
                for linha in linhas:
                    partes = linha.strip().split()
                    if partes:
                        try:
                            classe_id = int(float(partes[0]))
                            contagem_pasta[classe_id] += 1
                            contagem_total_geral[classe_id] += 1
                        except ValueError:
                            continue

        # Exibe os resultados mesclando ID da classe com o nome extraído do YAML
        print(f"\n📁 Dataset: {pasta} (Analisou {arquivos_analisados} arquivos .txt)")
        if not contagem_pasta:
            print("  -> Nenhuma anotação encontrada.")
        else:
            for class_id, count in sorted(contagem_pasta.items()):
                # Pega o nome no dicionário gerado pelo YAML, ou põe 'Desconhecido' se não achar
                nome_yaml = nomes_classes.get(class_id, "Desconhecido")
                print(f"  - Classe {class_id} (Nome no yaml: '{nome_yaml}'): {count} objetos")

    # ==========================================
    # RESUMO GERAL
    # ==========================================
    print("\n" + "="*60)
    print(" RESUMO GERAL DAS CLASSES (FUTURO DATASET DE TREINO)")
    print("="*60)
    if not contagem_total_geral:
        print("Nenhuma classe encontrada nas pastas fornecidas.")
    else:
        total_objetos = sum(contagem_total_geral.values())
        for class_id, count in sorted(contagem_total_geral.items()):
            nome_yaml = nomes_gerais_referencia.get(class_id, "Desconhecido")
            porcentagem = (count / total_objetos) * 100
            print(f"  - Classe {class_id} ('{nome_yaml}'): {count} objetos ({porcentagem:.1f}%)")
        print("-" * 60)
        print(f"  TOTAL GERAL DE OBJETOS: {total_objetos}")
    print("="*60 + "\n")


contar_objetos_com_nomes_do_yaml(PASTAS_ALVO)

 CONTAGEM DE OBJETOS MAPEANDO COM DATA.YAML

📁 Dataset: Recyclable Material Sorting ver2.v1i.yolov8 (Analisou 12553 arquivos .txt)
  - Classe 0 (Nome no yaml: 'metal'): 3533 objetos
  - Classe 1 (Nome no yaml: 'cardboard'): 4383 objetos
  - Classe 2 (Nome no yaml: 'glass'): 3935 objetos
  - Classe 3 (Nome no yaml: 'paper'): 1589 objetos
  - Classe 4 (Nome no yaml: 'plastic'): 6409 objetos

📁 Dataset: waste detection.v2i.yolov8 (Analisou 7939 arquivos .txt)
  - Classe 0 (Nome no yaml: 'CARDBOARD'): 4489 objetos
  - Classe 1 (Nome no yaml: 'GLASS'): 7435 objetos
  - Classe 2 (Nome no yaml: 'METAL'): 5613 objetos
  - Classe 3 (Nome no yaml: 'PAPER'): 4034 objetos
  - Classe 4 (Nome no yaml: 'PLASTIC'): 5650 objetos

📁 Dataset: waste detection.v8i.yolov8 (Analisou 2014 arquivos .txt)
  - Classe 0 (Nome no yaml: 'glass'): 544 objetos
  - Classe 1 (Nome no yaml: 'metal'): 463 objetos
  - Classe 2 (Nome no yaml: 'paper'): 624 objetos
  - Classe 3 (Nome no yaml: 'plastic'): 513 objetos

 RESUM

# Mapeando o nome correto das classes levando em conta o datset base

## Mapeando pasta "waste detection.v2i.yolov8"

In [4]:

# ==========================================
# CONFIGURAÇÕES
# ==========================================

PASTA_ALVO = 'waste detection.v2i.yolov8' 

# Splits possíveis gerados pelo Roboflow
SPLITS = ['train', 'valid', 'test']

# ==========================================
# DICIONÁRIO DE MAPEAMENTO DE CLASSES
# Formato -> Classe_Antiga: Classe_Nova
# ==========================================

# original ['CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']
# alvo ['metal', 'cardboard', 'glass', 'paper', 'plastic']

MAPEAMENTO = {
    0: 1,  
    1: 2,
    2: 0,
    3: 3,
    4: 4
}

def alterar_indices_classes(pasta_base, mapeamento):
    print("="*60)
    print(f" INICIANDO ALTERAÇÃO DE CLASSES EM: {pasta_base}")
    print("="*60)
    
    if not os.path.exists(pasta_base):
        print(f"[Erro] A pasta '{pasta_base}' não foi encontrada.")
        return

    arquivos_modificados_total = 0
    linhas_alteradas_total = 0

    for split in SPLITS:
        caminho_labels = os.path.join(pasta_base, split, 'labels')
        
        if not os.path.exists(caminho_labels):
            continue
            
        arquivos_txt = glob.glob(os.path.join(caminho_labels, '*.txt'))
        modificados_neste_split = 0
        
        for txt in arquivos_txt:
            linhas_modificadas = []
            arquivo_precisa_salvar = False
            
            with open(txt, 'r') as f:
                linhas = f.readlines()
                
            for linha in linhas:
                partes = linha.strip().split()
                
                if not partes:
                    continue
                
                try:
                    # O YOLO guarda a classe como o primeiro item da linha
                    cls_antiga = int(float(partes[0]))
                    
                    # Se a classe antiga está no nosso dicionário de mudança
                    if cls_antiga in mapeamento:
                        cls_nova = mapeamento[cls_antiga]
                        
                        # Só contabiliza se o número for realmente diferente
                        if cls_nova != cls_antiga:
                            partes[0] = str(cls_nova)
                            arquivo_precisa_salvar = True
                            linhas_alteradas_total += 1
                            
                    # Remonta a linha no formato YOLO (classe x y w h)
                    linha_nova = " ".join(partes) + "\n"
                    linhas_modificadas.append(linha_nova)
                    
                except ValueError:
                    # Caso tenha alguma linha mal formatada, mantém a original
                    linhas_modificadas.append(linha)
                    
            # Se houve alguma alteração real, sobrescreve o arquivo
            if arquivo_precisa_salvar:
                with open(txt, 'w') as f:
                    f.writelines(linhas_modificadas)
                modificados_neste_split += 1
                arquivos_modificados_total += 1

        if arquivos_txt:
            print(f" -> Pasta '{split}/labels': {modificados_neste_split} arquivos atualizados (de {len(arquivos_txt)} txts lidos).")

    print("\n" + "="*60)
    print(" RESUMO DA OPERAÇÃO")
    print("="*60)
    print(f"Total de arquivos .txt modificados: {arquivos_modificados_total}")
    print(f"Total de objetos (linhas) que tiveram a classe alterada: {linhas_alteradas_total}")
    print("="*60 + "\n")

# Executa a função
alterar_indices_classes(PASTA_ALVO, MAPEAMENTO)

 INICIANDO ALTERAÇÃO DE CLASSES EM: waste detection.v2i.yolov8
 -> Pasta 'train/labels': 3625 arquivos atualizados (de 5536 txts lidos).
 -> Pasta 'valid/labels': 1393 arquivos atualizados (de 1408 txts lidos).
 -> Pasta 'test/labels': 177 arquivos atualizados (de 995 txts lidos).

 RESUMO DA OPERAÇÃO
Total de arquivos .txt modificados: 5195
Total de objetos (linhas) que tiveram a classe alterada: 17537



## Mapeando pasta "waste detection.v8i.yolov8"

In [5]:

# ==========================================
# CONFIGURAÇÕES
# ==========================================

PASTA_ALVO = 'waste detection.v8i.yolov8' 

# Splits possíveis gerados pelo Roboflow
SPLITS = ['train', 'valid', 'test']

# ==========================================
# DICIONÁRIO DE MAPEAMENTO DE CLASSES
# Formato -> Classe_Antiga: Classe_Nova
# ==========================================

# original ['glass', 'metal', 'paper', 'plastic']
# alvo ['metal', 'cardboard', 'glass', 'paper', 'plastic']

MAPEAMENTO = {
    0: 2,  
    1: 0,
    2: 3,
    3: 4,
}

def alterar_indices_classes(pasta_base, mapeamento):
    print("="*60)
    print(f" INICIANDO ALTERAÇÃO DE CLASSES EM: {pasta_base}")
    print("="*60)
    
    if not os.path.exists(pasta_base):
        print(f"[Erro] A pasta '{pasta_base}' não foi encontrada.")
        return

    arquivos_modificados_total = 0
    linhas_alteradas_total = 0

    for split in SPLITS:
        caminho_labels = os.path.join(pasta_base, split, 'labels')
        
        if not os.path.exists(caminho_labels):
            continue
            
        arquivos_txt = glob.glob(os.path.join(caminho_labels, '*.txt'))
        modificados_neste_split = 0
        
        for txt in arquivos_txt:
            linhas_modificadas = []
            arquivo_precisa_salvar = False
            
            with open(txt, 'r') as f:
                linhas = f.readlines()
                
            for linha in linhas:
                partes = linha.strip().split()
                
                if not partes:
                    continue
                
                try:
                    # O YOLO guarda a classe como o primeiro item da linha
                    cls_antiga = int(float(partes[0]))
                    
                    # Se a classe antiga está no nosso dicionário de mudança
                    if cls_antiga in mapeamento:
                        cls_nova = mapeamento[cls_antiga]
                        
                        # Só contabiliza se o número for realmente diferente
                        if cls_nova != cls_antiga:
                            partes[0] = str(cls_nova)
                            arquivo_precisa_salvar = True
                            linhas_alteradas_total += 1
                            
                    # Remonta a linha no formato YOLO (classe x y w h)
                    linha_nova = " ".join(partes) + "\n"
                    linhas_modificadas.append(linha_nova)
                    
                except ValueError:
                    # Caso tenha alguma linha mal formatada, mantém a original
                    linhas_modificadas.append(linha)
                    
            # Se houve alguma alteração real, sobrescreve o arquivo
            if arquivo_precisa_salvar:
                with open(txt, 'w') as f:
                    f.writelines(linhas_modificadas)
                modificados_neste_split += 1
                arquivos_modificados_total += 1

        if arquivos_txt:
            print(f" -> Pasta '{split}/labels': {modificados_neste_split} arquivos atualizados (de {len(arquivos_txt)} txts lidos).")

    print("\n" + "="*60)
    print(" RESUMO DA OPERAÇÃO")
    print("="*60)
    print(f"Total de arquivos .txt modificados: {arquivos_modificados_total}")
    print(f"Total de objetos (linhas) que tiveram a classe alterada: {linhas_alteradas_total}")
    print("="*60 + "\n")

# Executa a função
alterar_indices_classes(PASTA_ALVO, MAPEAMENTO)

 INICIANDO ALTERAÇÃO DE CLASSES EM: waste detection.v8i.yolov8
 -> Pasta 'train/labels': 1799 arquivos atualizados (de 1799 txts lidos).
 -> Pasta 'valid/labels': 134 arquivos atualizados (de 135 txts lidos).
 -> Pasta 'test/labels': 80 arquivos atualizados (de 80 txts lidos).

 RESUMO DA OPERAÇÃO
Total de arquivos .txt modificados: 2013
Total de objetos (linhas) que tiveram a classe alterada: 2144



# Modificando os arquivos data.yaml
## Se foi modificado as listas de classes desses arquivos pela lista alvo ['metal', 'cardboard', 'glass', 'paper', 'plastic']

# Remoção de arquivos com lixo biodegradável de "waste detection.v2i.yolov8" já que essa classe não existe

In [7]:
# ==========================================
# CONFIGURAÇÕES
# ==========================================

PASTA_ALVO = 'waste detection.v2i.yolov8' 


TERMO_REMOVER = 'biodegradable'

SPLITS = ['train', 'valid', 'test']

def limpar_arquivos_por_termo(pasta_base, termo):
    print("="*60)
    print(f" INICIANDO FAXINA: Buscando arquivos com '{termo}'")
    print("="*60)
    
    if not os.path.exists(pasta_base):
        print(f"[Erro] A pasta '{pasta_base}' não foi encontrada.")
        return

    total_imagens_removidas = 0
    total_labels_removidos = 0

    for split in SPLITS:
        caminho_images = os.path.join(pasta_base, split, 'images')
        caminho_labels = os.path.join(pasta_base, split, 'labels')

        if not os.path.exists(caminho_images):
            continue

        # Busca qualquer imagem que contenha o termo no nome
        padrao_busca_img = os.path.join(caminho_images, f"*{termo}*.*")
        imagens_encontradas = glob.glob(padrao_busca_img)
        
        removidos_neste_split = 0

        for img_path in imagens_encontradas:
            nome_arquivo = os.path.basename(img_path)
            nome_base = os.path.splitext(nome_arquivo)[0]
            
            # Constrói o caminho de onde o .txt deveria estar
            txt_path = os.path.join(caminho_labels, f"{nome_base}.txt")

            # Remove a imagem
            try:
                os.remove(img_path)
                total_imagens_removidas += 1
                removidos_neste_split += 1
            except Exception as e:
                print(f"[Erro] Falha ao remover imagem {img_path}: {e}")

            # Remove o label correspondente (se existir)
            if os.path.exists(txt_path):
                try:
                    os.remove(txt_path)
                    total_labels_removidos += 1
                except Exception as e:
                    print(f"[Erro] Falha ao remover label {txt_path}: {e}")

        #  Busca de segurança (Remove labels órfãos que também tenham o termo no nome)
        if os.path.exists(caminho_labels):
            padrao_busca_txt = os.path.join(caminho_labels, f"*{termo}*.txt")
            txts_encontrados = glob.glob(padrao_busca_txt)
            
            for txt_path in txts_encontrados:
                try:
                    os.remove(txt_path)
                    total_labels_removidos += 1
                except Exception as e:
                    pass # Se já foi removido no passo anterior, apenas ignora

        if removidos_neste_split > 0:
            print(f" -> Pasta '{split}': Limpeza concluída.")

    # ==========================================
    # RELATÓRIO FINAL
    # ==========================================
    print("\n" + "="*60)
    print(" RESUMO DA LIMPEZA")
    print("="*60)
    if total_imagens_removidas == 0 and total_labels_removidos == 0:
        print(f"Nenhum arquivo contendo '{termo}' foi encontrado na pasta.")
    else:
        print(f"Total de imagens deletadas: {total_imagens_removidas}")
        print(f"Total de labels (.txt) deletados: {total_labels_removidos}")
    print("="*60 + "\n")

limpar_arquivos_por_termo(PASTA_ALVO, TERMO_REMOVER)

 INICIANDO FAXINA: Buscando arquivos com 'biodegradable'
 -> Pasta 'train': Limpeza concluída.
 -> Pasta 'valid': Limpeza concluída.

 RESUMO DA LIMPEZA
Total de imagens deletadas: 137
Total de labels (.txt) deletados: 137



# Analise após alterações para confirmar mudança

In [8]:
# ==========================================
# CONFIGURAÇÕES
# ==========================================
PASTAS_ALVO = [
    'Recyclable Material Sorting ver2.v1i.yolov8',  # Dataset Base com +12k imagens
    'waste detection.v2i.yolov8',                   # Dataset com mais imagens para complemento para todas as classes
    'waste detection.v8i.yolov8'                    # Dataset mais diversificado sem a classe de papelão
]

SPLITS = ['train', 'valid', 'test']

def contar_objetos_com_nomes_do_yaml(pastas):
    contagem_total_geral = defaultdict(int)
    nomes_gerais_referencia = {} # Guarda os nomes da primeira pasta válida para o resumo final

    print("="*60)
    print(" CONTAGEM DE OBJETOS MAPEANDO COM DATA.YAML")
    print("="*60)

    for pasta in pastas:
        contagem_pasta = defaultdict(int)
        arquivos_analisados = 0
        nomes_classes = {}
        
        if not os.path.exists(pasta):
            print(f"\n[!] Aviso: A pasta '{pasta}' não foi encontrada.")
            continue

        #  Busca e lê o arquivo data.yaml na raiz da pasta
        caminho_yaml = os.path.join(pasta, 'data.yaml')
        if os.path.exists(caminho_yaml):
            with open(caminho_yaml, 'r', encoding='utf-8') as f:
                dados_yaml = yaml.safe_load(f)
                # O Roboflow salva os nomes na chave 'names'
                names = dados_yaml.get('names', [])
                
                # Trata se for uma lista ou dicionário
                if isinstance(names, dict):
                    nomes_classes = names
                elif isinstance(names, list):
                    nomes_classes = {i: nome for i, nome in enumerate(names)}
                
                # Salva os nomes da primeira pasta para usar no resumo final
                if not nomes_gerais_referencia and nomes_classes:
                    nomes_gerais_referencia = nomes_classes
        else:
            print(f"\n[!] Aviso: data.yaml não encontrado em '{pasta}'. As classes ficarão sem nome.")

        # Conta os objetos nos arquivos .txt
        for split in SPLITS:
            caminho_labels = os.path.join(pasta, split, 'labels')
            if not os.path.exists(caminho_labels):
                continue
                
            arquivos_txt = glob.glob(os.path.join(caminho_labels, '*.txt'))
            arquivos_analisados += len(arquivos_txt)
            
            for txt in arquivos_txt:
                with open(txt, 'r') as f:
                    linhas = f.readlines()
                    
                for linha in linhas:
                    partes = linha.strip().split()
                    if partes:
                        try:
                            classe_id = int(float(partes[0]))
                            contagem_pasta[classe_id] += 1
                            contagem_total_geral[classe_id] += 1
                        except ValueError:
                            continue

        # Exibe os resultados mesclando ID da classe com o nome extraído do YAML
        print(f"\n📁 Dataset: {pasta} (Analisou {arquivos_analisados} arquivos .txt)")
        if not contagem_pasta:
            print("  -> Nenhuma anotação encontrada.")
        else:
            for class_id, count in sorted(contagem_pasta.items()):
                # Pega o nome no dicionário gerado pelo YAML, ou põe 'Desconhecido' se não achar
                nome_yaml = nomes_classes.get(class_id, "Desconhecido")
                print(f"  - Classe {class_id} (Nome no yaml: '{nome_yaml}'): {count} objetos")

    # ==========================================
    # RESUMO GERAL
    # ==========================================
    print("\n" + "="*60)
    print(" RESUMO GERAL DAS CLASSES (FUTURO DATASET DE TREINO)")
    print("="*60)
    if not contagem_total_geral:
        print("Nenhuma classe encontrada nas pastas fornecidas.")
    else:
        total_objetos = sum(contagem_total_geral.values())
        for class_id, count in sorted(contagem_total_geral.items()):
            nome_yaml = nomes_gerais_referencia.get(class_id, "Desconhecido")
            porcentagem = (count / total_objetos) * 100
            print(f"  - Classe {class_id} ('{nome_yaml}'): {count} objetos ({porcentagem:.1f}%)")
        print("-" * 60)
        print(f"  TOTAL GERAL DE OBJETOS: {total_objetos}")
    print("="*60 + "\n")


contar_objetos_com_nomes_do_yaml(PASTAS_ALVO)

 CONTAGEM DE OBJETOS MAPEANDO COM DATA.YAML

📁 Dataset: Recyclable Material Sorting ver2.v1i.yolov8 (Analisou 12553 arquivos .txt)
  - Classe 0 (Nome no yaml: 'metal'): 3533 objetos
  - Classe 1 (Nome no yaml: 'cardboard'): 4383 objetos
  - Classe 2 (Nome no yaml: 'glass'): 3935 objetos
  - Classe 3 (Nome no yaml: 'paper'): 1589 objetos
  - Classe 4 (Nome no yaml: 'plastic'): 6409 objetos

📁 Dataset: waste detection.v2i.yolov8 (Analisou 7802 arquivos .txt)
  - Classe 0 (Nome no yaml: 'metal'): 5503 objetos
  - Classe 1 (Nome no yaml: 'cardboard'): 4265 objetos
  - Classe 2 (Nome no yaml: 'glass'): 7369 objetos
  - Classe 3 (Nome no yaml: 'paper'): 3904 objetos
  - Classe 4 (Nome no yaml: 'plastic'): 5579 objetos

📁 Dataset: waste detection.v8i.yolov8 (Analisou 2014 arquivos .txt)
  - Classe 0 (Nome no yaml: 'metal'): 463 objetos
  - Classe 2 (Nome no yaml: 'glass'): 544 objetos
  - Classe 3 (Nome no yaml: 'paper'): 624 objetos
  - Classe 4 (Nome no yaml: 'plastic'): 513 objetos

 RESUM

# Juntando todos os arquivos em apenas 1 final Dataset_Unificado_YOLO

In [10]:
# ==========================================
# CONFIGURAÇÕES
# ==========================================
# Lista com todas as pastas 
PASTAS_ORIGEM = [
    'Recyclable Material Sorting ver2.v1i.yolov8', 
    'waste detection.v2i.yolov8',                  
    'waste detection.v8i.yolov8'
]

# Nome da nova pasta onde tudo será juntado
PASTA_DESTINO = 'Dataset_Unificado_YOLO'

SPLITS = ['train', 'valid', 'test']

def unificar_datasets(pastas_origem, pasta_destino):
    print("="*60)
    print(f" INICIANDO UNIFICAÇÃO PARA: {pasta_destino}")
    print("="*60)

    # Cria a estrutura de pastas do YOLOv8 na pasta de destino
    for split in SPLITS:
        os.makedirs(os.path.join(pasta_destino, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(pasta_destino, split, 'labels'), exist_ok=True)

    total_imagens_copiadas = 0
    primeiro_yaml_encontrado = None

    # Varre cada pasta original
    for pasta in pastas_origem:
        if not os.path.exists(pasta):
            print(f"[Aviso] Pasta '{pasta}' não encontrada. Pulando...")
            continue

        print(f"\nProcessando: {pasta}...")
        
        # Salva o caminho do primeiro data.yaml válido para copiar as classes depois
        if not primeiro_yaml_encontrado:
            caminho_yaml = os.path.join(pasta, 'data.yaml')
            if os.path.exists(caminho_yaml):
                primeiro_yaml_encontrado = caminho_yaml

        #  Processa cada split (train, valid, test)
        for split in SPLITS:
            nome_split_origem = split
            caminho_images = os.path.join(pasta, nome_split_origem, 'images')
            caminho_labels = os.path.join(pasta, nome_split_origem, 'labels')

            imagens = glob.glob(os.path.join(caminho_images, '*.*'))
            # Filtra apenas extensões de imagem comuns
            imagens = [img for img in imagens if img.lower().endswith(('.jpg', '.jpeg', '.png'))]
            
            copiadas_neste_split = 0

            for img_path in imagens:
                nome_arquivo = os.path.basename(img_path)
                nome_base = os.path.splitext(nome_arquivo)[0]
                
                # Procura o arquivo .txt correspondente
                txt_path = os.path.join(caminho_labels, f"{nome_base}.txt")
                
                # Só copia se existir a imagem E o label
                if os.path.exists(txt_path):
                    # Cria um novo nome seguro para evitar colisão (sobrescrever arquivos)
                    # Ex: 'Base_12k_imagem_001.jpg'
                    novo_nome_base = f"{pasta}_{nome_base}".replace(" ", "_")
                    novo_nome_img = f"{novo_nome_base}{os.path.splitext(img_path)[1]}"
                    novo_nome_txt = f"{novo_nome_base}.txt"
                    
                    destino_img = os.path.join(pasta_destino, split, 'images', novo_nome_img)
                    destino_txt = os.path.join(pasta_destino, split, 'labels', novo_nome_txt)
                    
                    # Copia os arquivos
                    shutil.copy2(img_path, destino_img)
                    shutil.copy2(txt_path, destino_txt)
                    
                    copiadas_neste_split += 1
                    total_imagens_copiadas += 1

            if copiadas_neste_split > 0:
                print(f"  -> {split}: {copiadas_neste_split} pares (imagem + txt) movidos.")

    # Gera o arquivo data.yaml final perfeitamente formatado para o Ultralytics
    print("\nGerando data.yaml unificado...")
    yaml_destino = os.path.join(pasta_destino, 'data.yaml')
    
    if primeiro_yaml_encontrado:
        with open(primeiro_yaml_encontrado, 'r', encoding='utf-8') as f:
            dados_antigos = yaml.safe_load(f)
            
        classes_nomes = dados_antigos.get('names', [])
        
        # Estrutura padrão que o Ultralytics YOLO não recusa
        novo_yaml = {
            'train': 'train/images',
            'val': 'valid/images',
            'test': 'test/images',
            'nc': len(classes_nomes),
            'names': classes_nomes
        }
        
        with open(yaml_destino, 'w', encoding='utf-8') as f:
            yaml.dump(novo_yaml, f, sort_keys=False, default_flow_style=False)
        print("  -> data.yaml criado com sucesso baseando-se no dataset base.")
    else:
        print("  -> [Aviso] Nenhum data.yaml encontrado nas origens. Você precisará criar um manualmente.")

    print("\n" + "="*60)
    print(" UNIFICAÇÃO CONCLUÍDA")
    print("="*60)
    print(f"Nova pasta criada: ./{pasta_destino}/")
    print(f"Total de pares (imagem + anotação) unificados: {total_imagens_copiadas}")
    print("="*60)

# Executa
unificar_datasets(PASTAS_ORIGEM, PASTA_DESTINO)

 INICIANDO UNIFICAÇÃO PARA: Dataset_Unificado_YOLO

Processando: Recyclable Material Sorting ver2.v1i.yolov8...
  -> train: 9794 pares (imagem + txt) movidos.
  -> valid: 1648 pares (imagem + txt) movidos.
  -> test: 1111 pares (imagem + txt) movidos.

Processando: waste detection.v2i.yolov8...
  -> train: 5446 pares (imagem + txt) movidos.
  -> valid: 1361 pares (imagem + txt) movidos.
  -> test: 995 pares (imagem + txt) movidos.

Processando: waste detection.v8i.yolov8...
  -> train: 1799 pares (imagem + txt) movidos.
  -> valid: 135 pares (imagem + txt) movidos.
  -> test: 80 pares (imagem + txt) movidos.

Gerando data.yaml unificado...
  -> data.yaml criado com sucesso baseando-se no dataset base.

 UNIFICAÇÃO CONCLUÍDA
Nova pasta criada: ./Dataset_Unificado_YOLO/
Total de pares (imagem + anotação) unificados: 22369


# Análise do Dataset final Dataset_Unificado_YOLO

In [11]:
# ==========================================
# CONFIGURAÇÕES
# ==========================================
PASTAS_ALVO = [
    'Dataset_Unificado_YOLO'                    
]

SPLITS = ['train', 'valid', 'test']

def contar_objetos_com_nomes_do_yaml(pastas):
    contagem_total_geral = defaultdict(int)
    nomes_gerais_referencia = {} # Guarda os nomes da primeira pasta válida para o resumo final

    print("="*60)
    print(" CONTAGEM DE OBJETOS MAPEANDO COM DATA.YAML")
    print("="*60)

    for pasta in pastas:
        contagem_pasta = defaultdict(int)
        arquivos_analisados = 0
        nomes_classes = {}
        
        if not os.path.exists(pasta):
            print(f"\n[!] Aviso: A pasta '{pasta}' não foi encontrada.")
            continue

        #  Busca e lê o arquivo data.yaml na raiz da pasta
        caminho_yaml = os.path.join(pasta, 'data.yaml')
        if os.path.exists(caminho_yaml):
            with open(caminho_yaml, 'r', encoding='utf-8') as f:
                dados_yaml = yaml.safe_load(f)
                # O Roboflow salva os nomes na chave 'names'
                names = dados_yaml.get('names', [])
                
                # Trata se for uma lista ou dicionário
                if isinstance(names, dict):
                    nomes_classes = names
                elif isinstance(names, list):
                    nomes_classes = {i: nome for i, nome in enumerate(names)}
                
                # Salva os nomes da primeira pasta para usar no resumo final
                if not nomes_gerais_referencia and nomes_classes:
                    nomes_gerais_referencia = nomes_classes
        else:
            print(f"\n[!] Aviso: data.yaml não encontrado em '{pasta}'. As classes ficarão sem nome.")

        # Conta os objetos nos arquivos .txt
        for split in SPLITS:
            caminho_labels = os.path.join(pasta, split, 'labels')
            if not os.path.exists(caminho_labels):
                continue
                
            arquivos_txt = glob.glob(os.path.join(caminho_labels, '*.txt'))
            arquivos_analisados += len(arquivos_txt)
            
            for txt in arquivos_txt:
                with open(txt, 'r') as f:
                    linhas = f.readlines()
                    
                for linha in linhas:
                    partes = linha.strip().split()
                    if partes:
                        try:
                            classe_id = int(float(partes[0]))
                            contagem_pasta[classe_id] += 1
                            contagem_total_geral[classe_id] += 1
                        except ValueError:
                            continue

        # Exibe os resultados mesclando ID da classe com o nome extraído do YAML
        print(f"\n📁 Dataset: {pasta} (Analisou {arquivos_analisados} arquivos .txt)")
        if not contagem_pasta:
            print("  -> Nenhuma anotação encontrada.")
        else:
            for class_id, count in sorted(contagem_pasta.items()):
                # Pega o nome no dicionário gerado pelo YAML, ou põe 'Desconhecido' se não achar
                nome_yaml = nomes_classes.get(class_id, "Desconhecido")
                print(f"  - Classe {class_id} (Nome no yaml: '{nome_yaml}'): {count} objetos")

    # ==========================================
    # RESUMO GERAL
    # ==========================================
    print("\n" + "="*60)
    print(" RESUMO GERAL DAS CLASSES (FUTURO DATASET DE TREINO)")
    print("="*60)
    if not contagem_total_geral:
        print("Nenhuma classe encontrada nas pastas fornecidas.")
    else:
        total_objetos = sum(contagem_total_geral.values())
        for class_id, count in sorted(contagem_total_geral.items()):
            nome_yaml = nomes_gerais_referencia.get(class_id, "Desconhecido")
            porcentagem = (count / total_objetos) * 100
            print(f"  - Classe {class_id} ('{nome_yaml}'): {count} objetos ({porcentagem:.1f}%)")
        print("-" * 60)
        print(f"  TOTAL GERAL DE OBJETOS: {total_objetos}")
    print("="*60 + "\n")


contar_objetos_com_nomes_do_yaml(PASTAS_ALVO)

 CONTAGEM DE OBJETOS MAPEANDO COM DATA.YAML

📁 Dataset: Dataset_Unificado_YOLO (Analisou 22369 arquivos .txt)
  - Classe 0 (Nome no yaml: 'metal'): 9499 objetos
  - Classe 1 (Nome no yaml: 'cardboard'): 8648 objetos
  - Classe 2 (Nome no yaml: 'glass'): 11848 objetos
  - Classe 3 (Nome no yaml: 'paper'): 6117 objetos
  - Classe 4 (Nome no yaml: 'plastic'): 12501 objetos

 RESUMO GERAL DAS CLASSES (FUTURO DATASET DE TREINO)
  - Classe 0 ('metal'): 9499 objetos (19.5%)
  - Classe 1 ('cardboard'): 8648 objetos (17.8%)
  - Classe 2 ('glass'): 11848 objetos (24.4%)
  - Classe 3 ('paper'): 6117 objetos (12.6%)
  - Classe 4 ('plastic'): 12501 objetos (25.7%)
------------------------------------------------------------
  TOTAL GERAL DE OBJETOS: 48613

